# Sistem Rekomendasi Berita Berbasis Konten (Content-Based Recommender System)
### Menggunakan Embedding Sentence-BERT (SBERT) & Cosine Similarity pada Dataset MIND-small

Notebook ini mengimplementasikan sistem rekomendasi berita berbasis konten (*Content-Based Recommender System*) menggunakan representasi embedding semantik Sentence-BERT (`all-MiniLM-L6-v2`) dan metrik kemiripan Cosine Similarity pada log aktivitas dataset MIND-small.
Dataset: https://msnews.github.io/


In [ ]:
try:
    import sentence_transformers
    import gradio
except ImportError:
    !pip install -q sentence-transformers gradio tqdm scikit-learn pandas numpy


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 50)
print(f"DEVICE YANG DIGUNAKAN: {device.upper()}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
else:
  print("=" * 50)


DEVICE YANG DIGUNAKAN: CUDA
GPU Model: Tesla T4


## 3. Dataset
Membaca berkas `news.tsv` dan `behaviors.tsv` secara langsung dari GitHub untuk subset Train dan Validation.


In [ ]:
# Mendefinisikan URL Raw GitHub untuk Dataset
BASE_URL = "https://raw.githubusercontent.com/nabila-azhari/recsys-2/master/data"

# Train
train_news_url = f"{BASE_URL}/MINDsmall_train/MINDsmall_train/news.tsv"
train_behaviors_url = f"{BASE_URL}/MINDsmall_train/MINDsmall_train/behaviors.tsv"

# Validation (Dev)
dev_news_url = f"{BASE_URL}/MINDsmall_dev/MINDsmall_dev/news.tsv"
dev_behaviors_url = f"{BASE_URL}/MINDsmall_dev/MINDsmall_dev/behaviors.tsv"

news_cols = ['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 'TitleEntities', 'AbstractEntities']
behaviors_cols = ['ImpressionID', 'UserID', 'Time', 'History', 'Impressions']
try:
    df_train_news = pd.read_csv(train_news_url, sep='	', names=news_cols, header=None)
    df_train_behaviors = pd.read_csv(train_behaviors_url, sep='	', names=behaviors_cols, header=None)

    df_dev_news = pd.read_csv(dev_news_url, sep='	', names=news_cols, header=None)
    df_dev_behaviors = pd.read_csv(dev_behaviors_url, sep='	', names=behaviors_cols, header=None)

except Exception as e:
    print(f"\nTerjadi kesalahan saat memuat data: {e}")
    print("Mencoba memuat dengan fallback branch 'main' jika branch default berbeda...")
    # Fallback jika default branch adalah 'main'
    ALT_BASE_URL = "https://raw.githubusercontent.com/nabila-azhari/recsys-2/main/data"
    train_news_url = f"{ALT_BASE_URL}/MINDsmall_train/MINDsmall_train/news.tsv"
    train_behaviors_url = f"{ALT_BASE_URL}/MINDsmall_train/MINDsmall_train/behaviors.tsv"
    dev_news_url = f"{ALT_BASE_URL}/MINDsmall_dev/MINDsmall_dev/news.tsv"
    dev_behaviors_url = f"{ALT_BASE_URL}/MINDsmall_dev/MINDsmall_dev/behaviors.tsv"

    df_train_news = pd.read_csv(train_news_url, sep='	', names=news_cols, header=None)
    df_train_behaviors = pd.read_csv(train_behaviors_url, sep='	', names=behaviors_cols, header=None)
    df_dev_news = pd.read_csv(dev_news_url, sep='	', names=news_cols, header=None)
    df_dev_behaviors = pd.read_csv(dev_behaviors_url, sep='	', names=behaviors_cols, header=None)
    print("\nDataset Train & Validation berhasil dimuat via Fallback!")


In [ ]:
# Statistik dasar dataset train
print("=== STATISTIK DATASET TRAIN ===")
print(f"Ukuran News Train      : {df_train_news.shape}")
print(f"Ukuran Behaviors Train : {df_train_behaviors.shape}")
print(f"Jumlah User Unik       : {df_train_behaviors['UserID'].nunique()}")
print(f"Jumlah Berita Unik     : {df_train_news['NewsID'].nunique()}")
print("-" * 40)

# Statistik dasar dataset validation (dev)
print("\n=== STATISTIK DATASET VALIDATION (DEV) ===")
print(f"Ukuran News Validation      : {df_dev_news.shape}")
print(f"Ukuran Behaviors Validation : {df_dev_behaviors.shape}")
print(f"Jumlah User Unik (Dev)      : {df_dev_behaviors['UserID'].nunique()}")
print(f"Jumlah Berita Unik (Dev)    : {df_dev_news['NewsID'].nunique()}")
print("-" * 40)

print("\n--- Contoh Data News (df_train_news) ---")
display(df_train_news.head(2))

print("\n--- Contoh Data Perilaku (df_train_behaviors) ---")
display(df_train_behaviors.head(2))


=== STATISTIK DATASET TRAIN ===
Ukuran News Train      : (51282, 8)
Ukuran Behaviors Train : (156965, 5)
Jumlah User Unik       : 50000
Jumlah Berita Unik     : 51282
----------------------------------------

=== STATISTIK DATASET VALIDATION (DEV) ===
Ukuran News Validation      : (42416, 8)
Ukuran Behaviors Validation : (73152, 5)
Jumlah User Unik (Dev)      : 50000
Jumlah Berita Unik (Dev)    : 42416
----------------------------------------

--- Contoh Data News (df_train_news) ---


,NewsID,Category,SubCategory,Title,Abstract,URL,TitleEntities,AbstractEntities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."



--- Contoh Data Perilaku (df_train_behaviors) ---


,ImpressionID,UserID,Time,History,Impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...


## 4. Preprocessing Data
Menggabungkan kolom `Title` dan `Abstract` menjadi satu kolom `content` untuk fitur teks utuh berita, serta membuang duplikat ID berita.


In [ ]:
def preprocess_news(df):
    df_cleaned = df.copy()

    # 1. Menangani missing value pada Title dan Abstract
    df_cleaned['Title'] = df_cleaned['Title'].fillna("")
    df_cleaned['Abstract'] = df_cleaned['Abstract'].fillna("")

    # 2. Menggabungkan Title dan Abstract sebagai representasi konten utuh
    df_cleaned['content'] = df_cleaned['Title'] + " " + df_cleaned['Abstract']

    # Menghapus berita yang duplikat dari ID yang sama agar proses pembuatan embedding efisien
    df_cleaned = df_cleaned.drop_duplicates(subset=['NewsID']).reset_index(drop=True)

    return df_cleaned

# Melakukan preprocessing pada dataset berita train dan dev
df_train_news_processed = preprocess_news(df_train_news)
df_dev_news_processed = preprocess_news(df_dev_news)

print("Preprocessing selesai!")
print(f"Jumlah baris berita train setelah cleaning: {len(df_train_news_processed)}")
print(f"Jumlah baris berita dev setelah cleaning  : {len(df_dev_news_processed)}")
print("\nContoh kolom 'content' hasil penggabungan:")
print(df_train_news_processed[['NewsID', 'content']].head(3).to_string(index=False))


Preprocessing selesai!
Jumlah baris berita train setelah cleaning: 51282
Jumlah baris berita dev setelah cleaning  : 42416

Contoh kolom 'content' hasil penggabungan:
NewsID                                                                                                                                                                                                                                                              content
N55528                                                                                                                     The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By Shop the notebooks, jackets, and more that the royals can't live without.
N19639                                                                                                                   50 Worst Habits For Belly Fat These seemingly harmless habits are holding you back and keeping you from shedding that unwanted belly fat for good.
N61837 The Cost of Trump's Ai

## 5. Ekstraksi Embedding Sentence-BERT
Menggunakan model `all-MiniLM-L6-v2` untuk mengonversi teks berita menjadi vektor representasi berdimensi 384.


In [ ]:
# 1. Inisialisasi Model Sentence-BERT
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# Menggabungkan berita unik dari train dan dev agar embedding-nya mencakup seluruh corpus berita
# Menggabungkan berita dari kedua subset dan membuang duplikat berdasarkan NewsID
all_news = pd.concat([df_train_news_processed, df_dev_news_processed]).drop_duplicates(subset=['NewsID']).reset_index(drop=True)
print(f"Total berita unik di seluruh corpus: {len(all_news)}")

# 2. Encoding konten berita menjadi embedding
news_contents = all_news['content'].tolist()
news_ids = all_news['NewsID'].tolist()
embeddings = model.encode(
    news_contents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

# 3. Menyimpan hasil embedding ke dalam lookup dictionary untuk pencarian cepat O(1)
news_embedding_dict = {news_ids[i]: embeddings[i] for i in range(len(news_ids))}

print("\nEncoding selesai!")
print(f"Ukuran matriks embedding: {embeddings.shape}")
print(f"Setiap berita kini direpresentasikan sebagai vektor {embeddings.shape[1]} dimensi.")


Mengunduh dan memuat model Sentence-BERT ('all-MiniLM-L6-v2') ke CUDA...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total berita unik di seluruh corpus: 65238


Batches:   0%|          | 0/1020 [00:00<?, ?it/s]


Encoding selesai!
Ukuran matriks embedding: (65238, 384)
Setiap berita kini direpresentasikan sebagai vektor 384 dimensi.


## 6. Pembentukan User Profile
Membentuk representasi preferensi pengguna (*User Profile*) dengan menghitung rata-rata (*mean centroid*) vektor embedding dari seluruh berita yang pernah dibaca pada kolom riwayat (`History`).


In [ ]:
def build_user_profile(user_id, df_behaviors, embedding_dict):
    """
    Fungsi untuk membangun User Profile berdasarkan rata-rata (centroid)
    embedding berita yang pernah dibaca di masa lalu.
    """
    # Mencari baris transaksi pengguna di behaviors
    user_rows = df_behaviors[df_behaviors['UserID'] == user_id]

    if user_rows.empty:
        return None

    # Mengambil baris pertama yang memuat data history (atau gabungkan jika ada lebih dari satu baris)
    # Menggabungkan semua histori unik dari user tersebut
    history_news_ids = []
    for history_str in user_rows['History'].dropna():
        history_news_ids.extend(history_str.split())

    # Hapus duplikat histori
    history_news_ids = list(set(history_news_ids))

    # Ambil embedding berita dari kamus lookup (jika tersedia)
    valid_embeddings = []
    for news_id in history_news_ids:
        if news_id in embedding_dict:
            valid_embeddings.append(embedding_dict[news_id])

    if not valid_embeddings:
        return None

    # Rata-rata (centroid) dari seluruh embedding berita di histori
    user_profile_vector = np.mean(valid_embeddings, axis=0)
    return user_profile_vector

# Uji coba membuat profil untuk satu user acak, contoh: U13740
test_user = "U13740"
profile = build_user_profile(test_user, df_train_behaviors, news_embedding_dict)

if profile is not None:
    print(f"User Profile untuk {test_user} berhasil dibuat!")
    print(f"Bentuk vektor User Profile: {profile.shape}")
    print(f"5 nilai pertama dari vektor profil: {profile[:5]}")
else:
    print(f"Gagal membuat User Profile untuk user {test_user}. Pastikan user memiliki histori klik.")


User Profile untuk U13740 berhasil dibuat!
Bentuk vektor User Profile: (384,)
5 nilai pertama dari vektor profil: [-0.01085217 -0.00882484  0.00497676  0.00929401  0.01916932]


## 7. Recommendation Engine
Menghitung tingkat kemiripan Cosine Similarity antara *User Profile* dengan seluruh berita kandidat, membuang berita yang sudah dibaca, dan menyajikan Top-K berita rekomendasi.


In [ ]:
# Menggabungkan seluruh metadata berita train & dev untuk memperkaya tampilan hasil rekomendasi
news_metadata = pd.concat([df_train_news_processed, df_dev_news_processed]).drop_duplicates(subset=['NewsID']).set_index('NewsID')

def recommend_news(user_id, df_behaviors, embedding_dict, metadata_df, top_k=10):
    """
    Fungsi untuk menghasilkan Top-K rekomendasi berita berdasarkan profil pengguna.
    """
    # 1. Membuat User Profile
    user_profile = build_user_profile(user_id, df_behaviors, embedding_dict)

    if user_profile is None:
        return f"Error: User ID {user_id} tidak ditemukan atau tidak memiliki histori membaca."

    # 2. Mengambil daftar berita yang sudah pernah dibaca oleh user agar bisa di-filter keluar
    user_rows = df_behaviors[df_behaviors['UserID'] == user_id]
    read_news_ids = set()
    for history_str in user_rows['History'].dropna():
        read_news_ids.update(history_str.split())

    # 3. Cosine similarity dengan seluruh berita di corpus
    candidate_news_ids = []
    candidate_embeddings = []

    for news_id, emb in embedding_dict.items():
        # Mengecualikan berita yang sudah pernah dibaca
        if news_id not in read_news_ids:
            candidate_news_ids.append(news_id)
            candidate_embeddings.append(emb)

    if not candidate_embeddings:
        return "Error: Tidak ada kandidat berita untuk direkomendasikan."

    candidate_embeddings = np.array(candidate_embeddings)
    user_profile_reshaped = user_profile.reshape(1, -1)

    # Cosine similarity
    similarities = cosine_similarity(user_profile_reshaped, candidate_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # 5. Dataframe hasil rekomendasi
    recommendations = []
    for rank, idx in enumerate(top_indices, 1):
        news_id = candidate_news_ids[idx]
        score = similarities[idx]
        title = metadata_df.loc[news_id, 'Title'] if news_id in metadata_df.index else "Tidak ada Judul"
        category = metadata_df.loc[news_id, 'Category'] if news_id in metadata_df.index else "Lainnya"

        recommendations.append({
            'Rank': rank,
            'NewsID': news_id,
            'Title': title,
            'Category': category,
            'Similarity Score': round(float(score), 4)
        })

    return pd.DataFrame(recommendations)

# Menguji coba untuk pengguna U13740
test_user = "U13740"
print(f"Menghasilkan Top-10 rekomendasi berita untuk user: {test_user}")
df_recommendations = recommend_news(test_user, df_train_behaviors, news_embedding_dict, news_metadata, top_k=10)

if isinstance(df_recommendations, pd.DataFrame):
    display(df_recommendations)
else:
    print(df_recommendations)


Menghasilkan Top-10 rekomendasi berita untuk user: U13740


,Rank,NewsID,Title,Category,Similarity Score
0,1,N52589,Exclusive: Hunter Biden on getting married aft...,news,0.5108
1,2,N55161,Best Response Ever From a 'Wheel of Fortune' C...,video,0.4715
2,3,N18069,Guy Who 'Doesn't Want Pets' Finally Gives In O...,lifestyle,0.4706
3,4,N2588,'It made it too real that we couldn't be here ...,news,0.4676
4,5,N37327,Howard Stern and Wife Beth Remarry After 11 Ye...,tv,0.4646
5,6,N35931,Crash on Swinton Avenue in Delray Beach sends ...,news,0.4599
6,7,N48290,Fond Farewell For Alex Denis,news,0.4530
7,8,N30129,Hunter Biden and wife Melissa Biden on weather...,news,0.4523
8,9,N60607,Teresa and Joe Giudice: Biggest Bombshells Fro...,tv,0.4511
9,10,N26374,SEE IT: Christopher Johnson Critical Of Jets I...,sports,0.4490


## 8. Evaluasi Performa
Mengukur kinerja sistem rekomendasi menggunakan data validasi (`MINDsmall_dev`) dengan metrik **Precision@K** dan **Recall@K**.


In [ ]:
def evaluate_user_recommendation(user_id, history_df, impression_news_list, clicked_news_set, embedding_dict, K=5):
    """
    Menghitung Precision@K dan Recall@K untuk satu transaksi impresi user di data validation.
    """
    # 1. Membangun User Profile dari histori membaca (train behavior)
    user_profile = build_user_profile(user_id, history_df, embedding_dict)

    if user_profile is None or not clicked_news_set:
        return None

    # 2. Similarity untuk berita yang ada di daftar Impression
    candidate_embeddings = []
    valid_candidates = []

    for news_id in impression_news_list:
        if news_id in embedding_dict:
            candidate_embeddings.append(embedding_dict[news_id])
            valid_candidates.append(news_id)

    if not valid_candidates:
        return None

    # Cosine similarity
    candidate_embeddings = np.array(candidate_embeddings)
    user_profile_reshaped = user_profile.reshape(1, -1)
    similarities = cosine_similarity(user_profile_reshaped, candidate_embeddings)[0]
    sorted_indices = np.argsort(similarities)[::-1]
    top_k_recommendations = [valid_candidates[idx] for idx in sorted_indices[:K]]

    # 4. Metrik evaluasi
    recommended_set = set(top_k_recommendations)
    correct_recommendations = recommended_set.intersection(clicked_news_set)

    precision_k = len(correct_recommendations) / K
    recall_k = len(correct_recommendations) / len(clicked_news_set)

    return precision_k, recall_k

# Evaluasi
precision_scores = []
recall_scores = []
evaluated_count = 0
K_VAL = 5
max_eval_users = 500

for idx, row in tqdm(df_dev_behaviors.iterrows(), total=min(len(df_dev_behaviors), max_eval_users)):
    user_id = row['UserID']
    impressions_str = row['Impressions']

    if pd.isna(impressions_str) or pd.isna(row['History']):
        continue


    impression_items = impressions_str.split()

    impression_news_list = []
    clicked_news_set = set()

    for item in impression_items:
        if '-' in item:
            news_id, click_status = item.split('-')
            impression_news_list.append(news_id)
            if click_status == '1':
                clicked_news_set.add(news_id)

    if not clicked_news_set:
        continue

    # Metrik
    metrics = evaluate_user_recommendation(
        user_id=user_id,
        history_df=df_train_behaviors,
        impression_news_list=impression_news_list,
        clicked_news_set=clicked_news_set,
        embedding_dict=news_embedding_dict,
        K=K_VAL
    )

    if metrics is not None:
        p_k, r_k = metrics
        precision_scores.append(p_k)
        recall_scores.append(r_k)
        evaluated_count += 1

    if evaluated_count >= max_eval_users:
        break

#Rata-rata hasil evaluasi
avg_precision = np.mean(precision_scores) if precision_scores else 0
avg_recall = np.mean(recall_scores) if recall_scores else 0

print("=" * 60)
print(f"HASIL EVALUASI MODEL (K = {K_VAL}) PADA {evaluated_count} USER AKTIF:")
print("-" * 60)
print(f"Rata-rata Precision@{K_VAL} : {avg_precision:.4f} ({round(avg_precision * 100, 2)}%)")
print(f"Rata-rata Recall@{K_VAL}    : {avg_recall:.4f} ({round(avg_recall * 100, 2)}%)")
print("=" * 60)
print("Penjelasan hasil:")
print(f"- Rata-rata Precision@{K_VAL} sebesar {avg_precision:.4f} menunjukkan bahwa sekitar {round(avg_precision * K_VAL, 2)} dari {K_VAL} berita yang disajikan sistem sesuai dengan ketertarikan nyata pengguna.")
print(f"- Rata-rata Recall@{K_VAL} sebesar {avg_recall:.4f} menandakan sistem berhasil menemukan {round(avg_recall * 100, 1)}% dari total keseluruhan berita yang diminati pengguna dalam satu sesi impresi.")


  0%|          | 0/500 [00:00<?, ?it/s]

HASIL EVALUASI MODEL (K = 5) PADA 500 USER AKTIF:
------------------------------------------------------------
Rata-rata Precision@5 : 0.1220 (12.2%)
Rata-rata Recall@5    : 0.4888 (48.88%)
Penjelasan hasil:
- Rata-rata Precision@5 sebesar 0.1220 menunjukkan bahwa sekitar 0.61 dari 5 berita yang disajikan sistem sesuai dengan ketertarikan nyata pengguna.
- Rata-rata Recall@5 sebesar 0.4888 menandakan sistem berhasil menemukan 48.9% dari total keseluruhan berita yang diminati pengguna dalam satu sesi impresi.


## 9. User Interface (Gradio)

Untuk mempermudah penggunaan sistem rekomendasi secara praktis dan memberikan demonstrasi interaktif bagi pengguna umum maupun dosen penguji, kita membangun **User Interface (UI)** sederhana dan elegan menggunakan **Gradio**.

Dengan mengeksekusi dua cell di bawah ini:
1. Cell pertama menggunakan perintah magic `%%writefile` untuk **secara otomatis men-generate berkas `run_gradio.py`** secara lokal di komputer Anda. Berkas ini dapat dijalankan langsung di terminal kapan saja menggunakan perintah `python run_gradio.py`.
2. Cell kedua menggunakan perintah magic `%run -i` untuk **menjalankan berkas `run_gradio.py` secara langsung dan menampilkan antarmuka web interaktif secara inline** di dalam notebook ini!


In [ ]:
%%writefile run_gradio.py
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

# 1. Setup Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Setup Dataset URL
BASE_URL = "https://raw.githubusercontent.com/nabila-azhari/recsys-2/master/data"
train_news_url = f"{BASE_URL}/MINDsmall_train/MINDsmall_train/news.tsv"
train_behaviors_url = f"{BASE_URL}/MINDsmall_train/MINDsmall_train/behaviors.tsv"
dev_news_url = f"{BASE_URL}/MINDsmall_dev/MINDsmall_dev/news.tsv"

news_cols = ['NewsID', 'Category', 'SubCategory', 'Title', 'Abstract', 'URL', 'TitleEntities', 'AbstractEntities']
behaviors_cols = ['ImpressionID', 'UserID', 'Time', 'History', 'Impressions']

# PERBAIKAN: Mengubah sep=' ' menjadi sep='\t' karena format file adalah .tsv (Tab Separated Values)
df_train_news = pd.read_csv(train_news_url, sep='\t', names=news_cols, header=None)
df_train_behaviors = pd.read_csv(train_behaviors_url, sep='\t', names=behaviors_cols, header=None)
df_dev_news = pd.read_csv(dev_news_url, sep='\t', names=news_cols, header=None)

# Preprocessing
def preprocess_news(df):
    df_cleaned = df.copy()
    df_cleaned['Title'] = df_cleaned['Title'].fillna("")
    df_cleaned['Abstract'] = df_cleaned['Abstract'].fillna("")
    df_cleaned['content'] = df_cleaned['Title'] + " " + df_cleaned['Abstract']
    df_cleaned = df_cleaned.drop_duplicates(subset=['NewsID']).reset_index(drop=True)
    return df_cleaned

df_train_news_processed = preprocess_news(df_train_news)
df_dev_news_processed = preprocess_news(df_dev_news)

all_news = pd.concat([df_train_news_processed, df_dev_news_processed]).drop_duplicates(subset=['NewsID']).reset_index(drop=True)
news_metadata = all_news.set_index('NewsID')

# Sentence-BERT Embeddings
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
news_contents = all_news['content'].tolist()
news_ids = all_news['NewsID'].tolist()

embeddings = model.encode(news_contents, batch_size=64, show_progress_bar=False, convert_to_numpy=True)
news_embedding_dict = {news_ids[i]: embeddings[i] for i in range(len(news_ids))}

# Recommendation Logic
def build_user_profile(user_id, df_behaviors, embedding_dict):
    user_rows = df_behaviors[df_behaviors['UserID'] == user_id]
    if user_rows.empty:
        return None
    history_news_ids = []
    for history_str in user_rows['History'].dropna():
        history_news_ids.extend(history_str.split())
    history_news_ids = list(set(history_news_ids))
    valid_embeddings = []
    for news_id in history_news_ids:
        if news_id in embedding_dict:
            valid_embeddings.append(embedding_dict[news_id])
    if not valid_embeddings:
        return None
    return np.mean(valid_embeddings, axis=0)

def recommend_news(user_id, df_behaviors, embedding_dict, metadata_df, top_k=10):
    user_profile = build_user_profile(user_id, df_behaviors, embedding_dict)
    if user_profile is None:
        return f"Error: User ID '{user_id}' tidak memiliki riwayat klik (History) di dalam database latihan."
    user_rows = df_behaviors[df_behaviors['UserID'] == user_id]
    read_news_ids = set()
    for history_str in user_rows['History'].dropna():
        read_news_ids.update(history_str.split())
    candidate_news_ids = []
    candidate_embeddings = []
    for news_id, emb in embedding_dict.items():
        if news_id not in read_news_ids:
            candidate_news_ids.append(news_id)
            candidate_embeddings.append(emb)
    if not candidate_embeddings:
        return "Error: Tidak ada berita kandidat baru yang tersedia."
    candidate_embeddings = np.array(candidate_embeddings)
    user_profile_reshaped = user_profile.reshape(1, -1)
    similarities = cosine_similarity(user_profile_reshaped, candidate_embeddings)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    recommendations = []
    for rank, idx in enumerate(top_indices, 1):
        news_id = candidate_news_ids[idx]
        score = similarities[idx]
        title = metadata_df.loc[news_id, 'Title'] if news_id in metadata_df.index else "Tidak ada Judul"
        category = metadata_df.loc[news_id, 'Category'] if news_id in metadata_df.index else "Lainnya"
        recommendations.append({
            'Rank': rank,
            'NewsID': news_id,
            'Title': title,
            'Category': category,
            'Similarity Score': round(float(score), 4)
        })
    return pd.DataFrame(recommendations)

def gradio_recommend(user_id, top_k):
    user_id = str(user_id).strip()
    if not user_id:
        return gr.update(visible=False), "Mohon masukkan User ID terlebih dahulu."
    res = recommend_news(user_id=user_id, df_behaviors=df_train_behaviors, embedding_dict=news_embedding_dict, metadata_df=news_metadata, top_k=int(top_k))
    if isinstance(res, str):
        return gr.update(visible=False), res
    return gr.update(value=res, visible=True), f"Menampilkan {top_k} rekomendasi berita untuk User ID: {user_id}"

# UI Layout (Clean & Minimalist Base Theme)
with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown(
        """
        # Sistem Rekomendasi Berita
        Sistem ini memberikan rekomendasi berita personal berdasarkan riwayat membaca pengguna.
        """
    )
    with gr.Row():
        with gr.Column(scale=1):
            user_input = gr.Textbox(label="User ID", placeholder="Masukkan ID Pengguna...", value="U13740")
            top_k_slider = gr.Slider(label="Jumlah Berita (K)", minimum=3, maximum=20, step=1, value=10)
            btn = gr.Button("Tampilkan Rekomendasi", variant="primary")
            gr.Markdown(
                """
                **Contoh User ID Aktif:**
                * U13740
                * U91836
                * U84444
                """
            )
        with gr.Column(scale=2):
            status_output = gr.Markdown("Masukkan User ID di kolom kiri lalu klik tombol untuk melihat rekomendasi.")
            results_table = gr.Dataframe(headers=["Rank", "NewsID", "Title", "Category", "Similarity Score"], datatype=["str", "str", "str", "str", "number"], visible=False)
    btn.click(fn=gradio_recommend, inputs=[user_input, top_k_slider], outputs=[results_table, status_output])

demo.launch(share=True)

Overwriting run_gradio.py


In [ ]:
!python run_gradio.py

Loading weights: 100% 103/103 [00:00<00:00, 1964.05it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/content/run_gradio.py:111: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base()) as demo:
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d73ba4c7377227a5e2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main 